# 04 — Render

**Purpose:** Build Plotly charts from model outputs and export a static HTML dashboard to `docs/` for GitHub Pages.

## Charts
| Chart | Data |
|-------|------|
| Yield curve spread (10y-2y) time series | `indicators.parquet` |
| Recession probability gauge + history | `indicators.parquet` |
| Inflation regime heatmap | `indicators.parquet` |
| Global growth pulse bar chart | `indicators.parquet` |
| Risk-on / risk-off dial | `latest_snapshot.json` |
| Key metrics header | `latest_snapshot.json` |

## Outputs
- `docs/index.html` — self-contained HTML dashboard (Plotly CDN)

## Papermill Parameters
- `run_date` — ISO date string injected by the GitHub Actions workflow

In [ ]:
run_date = None

In [ ]:
# Mount Google Drive for persistent storage (Colab only)
try:
    from google.colab import drive
    from pathlib import Path
    drive.mount("/content/drive")
    DRIVE_DATA = Path("/content/drive/MyDrive/macro-dashboard/data")
    DRIVE_DOCS = Path("/content/drive/MyDrive/macro-dashboard/docs")
    DRIVE_DOCS.mkdir(parents=True, exist_ok=True)
    _IN_COLAB = True
    print("Drive mounted.")
except Exception:
    _IN_COLAB = False
    print("Not in Colab — using local directories.")

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
    "pandas", "numpy", "plotly", "pyarrow"])
print("Packages ready.")

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import plotly.graph_objects as go
from plotly.subplots import make_subplots

OUTPUTS_DIR = DRIVE_DATA / "outputs" if _IN_COLAB else Path("data/outputs")
DOCS_DIR    = DRIVE_DOCS             if _IN_COLAB else Path("docs")
DOCS_DIR.mkdir(parents=True, exist_ok=True)
print(f"OUTPUTS_DIR : {OUTPUTS_DIR}")
print(f"DOCS_DIR    : {DOCS_DIR}")

In [ ]:
ind  = pd.read_parquet(OUTPUTS_DIR / "indicators.parquet")
snap = json.loads((OUTPUTS_DIR / "latest_snapshot.json").read_text())

ind.index = pd.to_datetime(ind.index)
# Trim to post-1990 for cleaner charts
ind = ind[ind.index >= "1990-01-01"]

print("Indicators loaded:", ind.shape, "| as_of:", snap["as_of"])
print(json.dumps(snap, indent=2))

In [ ]:
DARK = "plotly_dark"
CYAN, ORANGE, RED, GREEN, GREY = "#00bcd4", "#ff9800", "#ef5350", "#66bb6a", "#78909c"

# ── 1. Yield Curve Spread ──────────────────────────────────────────────────
fig_curve = go.Figure()
spread = ind["yield_spread_10y2y"].dropna()
fig_curve.add_trace(go.Scatter(
    x=spread.index, y=spread.values,
    mode="lines", name="10y–2y Spread",
    line=dict(color=CYAN, width=1.5),
    fill="tozeroy",
    fillcolor="rgba(0,188,212,0.08)"
))
fig_curve.add_hline(y=0, line=dict(color=RED, dash="dash", width=1))
fig_curve.update_layout(
    template=DARK, title="Yield Curve Spread (10y – 2y)",
    yaxis_title="% points", height=350,
    margin=dict(l=50, r=20, t=50, b=40)
)

# ── 2. Recession Probability ───────────────────────────────────────────────
fig_rec = go.Figure()
rec = ind["recession_prob"].dropna()
fig_rec.add_trace(go.Scatter(
    x=rec.index, y=rec.values * 100,
    mode="lines", name="Recession Prob",
    line=dict(color=ORANGE, width=1.5),
    fill="tozeroy", fillcolor="rgba(255,152,0,0.1)"
))
fig_rec.add_hline(y=30, line=dict(color=RED, dash="dot", width=1),
                  annotation_text="30% threshold", annotation_position="bottom right")
fig_rec.update_layout(
    template=DARK, title="Recession Probability — 12m Ahead (Estrella-Mishkin)",
    yaxis_title="%", yaxis_range=[0, 100], height=350,
    margin=dict(l=50, r=20, t=50, b=40)
)

# ── 3. Inflation Regime ────────────────────────────────────────────────────
fig_inf = go.Figure()
zscore = ind["inflation_zscore"].dropna()
colors = [RED if v > 1.5 else ORANGE if v > 0.5 else GREEN if v < -0.5 else GREY
          for v in zscore.values]
fig_inf.add_trace(go.Bar(
    x=zscore.index, y=zscore.values,
    marker_color=colors, name="Inflation Z-Score"
))
fig_inf.add_hline(y=1.5,  line=dict(color=RED,    dash="dot", width=1))
fig_inf.add_hline(y=-0.5, line=dict(color=GREEN,  dash="dot", width=1))
fig_inf.update_layout(
    template=DARK, title="Inflation Regime (Z-Score vs 20yr Rolling Avg)",
    yaxis_title="Z-Score", height=350,
    margin=dict(l=50, r=20, t=50, b=40)
)

# ── 4. Risk Score Gauge ────────────────────────────────────────────────────
risk_val = snap["risk_score"] or 0
fig_gauge = go.Figure(go.Indicator(
    mode="gauge+number+delta",
    value=risk_val,
    delta={"reference": 0},
    title={"text": "Risk-On / Risk-Off Score", "font": {"size": 14}},
    gauge={
        "axis": {"range": [-1, 1], "tickwidth": 1},
        "bar": {"color": GREEN if risk_val > 0 else RED},
        "steps": [
            {"range": [-1, -0.3], "color": "rgba(239,83,80,0.3)"},
            {"range": [-0.3, 0.3], "color": "rgba(120,144,156,0.2)"},
            {"range": [0.3, 1],   "color": "rgba(102,187,106,0.3)"},
        ],
        "threshold": {"line": {"color": "white", "width": 2}, "value": risk_val}
    }
))
fig_gauge.update_layout(template=DARK, height=350,
                        margin=dict(l=30, r=30, t=60, b=30))

# ── 5. Key Metrics Card ────────────────────────────────────────────────────
regime_labels = {-1: "Low", 0: "Normal", 1: "Elevated", 2: "Very High"}
metrics_html = f"""
<div style="font-family:monospace;background:#1e1e2e;color:#cdd6f4;
            padding:24px;border-radius:8px;line-height:2">
  <h2 style="color:#89b4fa;margin-top:0">Global Macro Snapshot — {snap['as_of']}</h2>
  <table style="width:100%;border-collapse:collapse">
    <tr><td>Yield Spread 10y–2y</td><td style="color:{GREEN if (snap['yield_spread_10y2y'] or 0)>0 else RED}">
        {snap['yield_spread_10y2y']:+.2f}%</td></tr>
    <tr><td>Yield Spread 10y–3m</td><td style="color:{GREEN if (snap['yield_spread_10y3m'] or 0)>0 else RED}">
        {snap['yield_spread_10y3m']:+.2f}%</td></tr>
    <tr><td>Inversion Signal</td><td>{'🔴 YES' if snap['inversion_signal'] else '🟢 NO'}
        ({snap['months_inverted']} months)</td></tr>
    <tr><td>Recession Probability</td><td style="color:{RED if (snap['recession_prob'] or 0)>0.3 else ORANGE}">
        {(snap['recession_prob'] or 0)*100:.1f}%</td></tr>
    <tr><td>Inflation Regime</td><td>{regime_labels.get(snap['inflation_regime'], 'Normal')}
        (z={snap['inflation_zscore']})</td></tr>
    <tr><td>Global Growth Pulse</td><td>{f"{snap['global_growth_pulse']:.1f}%" if snap['global_growth_pulse'] else 'N/A'}</td></tr>
    <tr><td>Risk Score</td><td style="color:{GREEN if (snap['risk_score'] or 0)>0 else RED}">
        {snap['risk_score']:+.2f} ({'Risk-On' if (snap['risk_score'] or 0)>0 else 'Risk-Off'})</td></tr>
  </table>
</div>"""

print("All figures built.")
fig_curve.show()
fig_rec.show()
fig_inf.show()
fig_gauge.show()

In [ ]:
# ── 6. Country Scoreboard Table ───────────────────────────────────────────
sb = pd.read_parquet(OUTPUTS_DIR / "country_scoreboard.parquet")

def _f(v, spec, na="N/A"):
    try:
        f = float(v)
        return na if np.isnan(f) else spec.format(f)
    except Exception:
        return na

def _c(v, lo, hi, invert=False):
    try:
        f = float(v)
        if np.isnan(f): return "rgba(120,144,156,0.15)"
    except Exception:
        return "rgba(120,144,156,0.15)"
    if not invert:
        if f >= hi: return "rgba(102,187,106,0.35)"
        if f >= lo: return "rgba(255,152,0,0.35)"
        return "rgba(239,83,80,0.35)"
    else:
        if f <= lo: return "rgba(102,187,106,0.35)"
        if f <= hi: return "rgba(255,152,0,0.35)"
        return "rgba(239,83,80,0.35)"

n = len(sb)
NEUTRAL = "rgba(49,50,68,0.8)"

columns = [
    ("gdp_actual",      lambda v: _f(v, "{:+.1f}"),  lambda v: _c(v, 0, 2)),
    ("gdp_forecast",    lambda v: _f(v, "{:+.1f}"),  lambda v: _c(v, 0, 2)),
    ("inflation",       lambda v: _f(v, "{:.1f}"),   lambda v: _c(v, 3, 5, invert=True)),
    ("unemployment",    lambda v: _f(v, "{:.1f}"),   lambda v: _c(v, 5, 8, invert=True)),
    ("current_account", lambda v: _f(v, "{:+.1f}"),  lambda v: _c(v, -3, 0)),
    ("govt_debt",       lambda v: _f(v, "{:.0f}"),   lambda v: _c(v, 60, 90, invert=True)),
    ("policy_rate",     lambda v: _f(v, "{:.2f}"),   lambda v: NEUTRAL),
    ("stock_ytd",       lambda v: _f(v, "{:+.1f}"),  lambda v: _c(v, -10, 0)),
]

countries    = sb.index.tolist()
cell_values  = [countries]
cell_colors  = [["rgba(30,30,46,0.95)"] * n]

for col, fmt_fn, clr_fn in columns:
    vals = sb[col].tolist()
    cell_values.append([fmt_fn(v) for v in vals])
    cell_colors.append([clr_fn(v) for v in vals])

fig_scoreboard = go.Figure(go.Table(
    columnwidth=[145, 75, 80, 68, 78, 88, 88, 75, 80],
    header=dict(
        values=[
            "<b>Country</b>",
            "<b>GDP %</b><br>2024",
            "<b>GDP %</b><br>forecast",
            "<b>CPI %</b>",
            "<b>Unemp %</b>",
            "<b>Curr Acct</b><br>% GDP",
            "<b>Govt Debt</b><br>% GDP",
            "<b>Policy</b><br>Rate %",
            "<b>Stock</b><br>YTD %",
        ],
        fill_color="#313244",
        font=dict(color="#cdd6f4", size=12),
        align="center",
        height=44,
        line=dict(color="#45475a", width=1),
    ),
    cells=dict(
        values=cell_values,
        fill_color=cell_colors,
        font=dict(color="#cdd6f4", size=12),
        align=["left"] + ["center"] * 8,
        height=34,
        line=dict(color="#313244", width=1),
    )
))
fig_scoreboard.update_layout(
    template=DARK,
    title="Country Scoreboard — Key Economic Indicators",
    height=540,
    margin=dict(l=10, r=10, t=50, b=10),
)
fig_scoreboard.show()
print("Scoreboard figure built.")

In [ ]:
# Assemble self-contained HTML dashboard with tab navigation
curve_html      = fig_curve.to_html(full_html=False, include_plotlyjs="cdn")
rec_html        = fig_rec.to_html(full_html=False,   include_plotlyjs=False)
inf_html        = fig_inf.to_html(full_html=False,   include_plotlyjs=False)
gauge_html      = fig_gauge.to_html(full_html=False, include_plotlyjs=False)
scoreboard_html = fig_scoreboard.to_html(full_html=False, include_plotlyjs=False)

html = f"""<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Global Macro Dashboard</title>
  <style>
    body     {{ margin:0; background:#11111b; color:#cdd6f4; font-family:system-ui,sans-serif; }}
    h1       {{ text-align:center; color:#89b4fa; padding:24px 0 0; margin:0; font-size:1.6rem; }}
    p.sub    {{ text-align:center; color:#6c7086; margin:4px 0 0; font-size:.9rem; }}
    .tab-bar {{ display:flex; gap:0; padding:16px 16px 0; border-bottom:2px solid #313244; }}
    .tab     {{ background:transparent; color:#a6adc8; border:none;
               border-bottom:2px solid transparent; margin-bottom:-2px;
               padding:10px 28px; cursor:pointer; font-size:.95rem;
               font-family:system-ui,sans-serif; }}
    .tab:hover  {{ color:#cdd6f4; }}
    .tab.active {{ color:#89b4fa; border-bottom-color:#89b4fa; }}
    .tab-pane   {{ display:none; }}
    .tab-pane.active {{ display:block; }}
    .grid    {{ display:grid; grid-template-columns:1fr 1fr; gap:16px; padding:16px; }}
    .card    {{ background:#1e1e2e; border-radius:8px; padding:8px; }}
    .full    {{ grid-column:1/-1; }}
    .sb-wrap {{ padding:8px 16px 16px; }}
    .legend  {{ display:flex; gap:20px; justify-content:flex-end;
               padding:12px 16px 0; font-size:.8rem; color:#6c7086; align-items:center; }}
    .dot     {{ width:10px; height:10px; border-radius:50%;
               display:inline-block; margin-right:5px; vertical-align:middle; }}
  </style>
</head>
<body>
  <h1>Global Macro Dashboard</h1>
  <p class="sub">As of {snap['as_of']} &middot; Powered by FRED &middot; World Bank &middot; IMF WEO</p>

  <div class="tab-bar">
    <button class="tab active" onclick="switchTab(event,'macro')">Macro Signals</button>
    <button class="tab" onclick="switchTab(event,'scoreboard')">Country Scoreboard</button>
  </div>

  <div id="macro" class="tab-pane active">
    <div class="grid">
      <div class="card full">{metrics_html}</div>
      <div class="card">{curve_html}</div>
      <div class="card">{rec_html}</div>
      <div class="card">{inf_html}</div>
      <div class="card">{gauge_html}</div>
    </div>
  </div>

  <div id="scoreboard" class="tab-pane">
    <div class="legend">
      <span><span class="dot" style="background:rgba(102,187,106,0.9)"></span>Positive</span>
      <span><span class="dot" style="background:rgba(255,152,0,0.9)"></span>Caution</span>
      <span><span class="dot" style="background:rgba(239,83,80,0.9)"></span>Negative</span>
    </div>
    <div class="sb-wrap">{scoreboard_html}</div>
  </div>

  <script>
    function switchTab(e, id) {{
      document.querySelectorAll('.tab-pane').forEach(p => p.classList.remove('active'));
      document.querySelectorAll('.tab').forEach(b => b.classList.remove('active'));
      document.getElementById(id).classList.add('active');
      e.currentTarget.classList.add('active');
    }}
  </script>
</body>
</html>"""

out = DOCS_DIR / "index.html"
out.write_text(html, encoding="utf-8")
print(f"Dashboard saved: {out}")
print(f"Size: {len(html):,} bytes")

In [ ]:
# Push docs/index.html to GitHub
# Reads GITHUB_TOKEN from Colab Secrets — never paste tokens in code.
# To add the secret: click the key icon (🔑) in the Colab left sidebar
# → New secret → Name: GITHUB_TOKEN, Value: your token
import requests, base64

REPO   = "trevmon28/macro-dashboard"
BRANCH = "master"
FILE   = "docs/index.html"

try:
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
    if not TOKEN:
        raise ValueError("GITHUB_TOKEN secret is empty")
except Exception as e:
    raise EnvironmentError(
        "Add GITHUB_TOKEN to Colab Secrets: key icon → New secret → GITHUB_TOKEN"
    ) from e

headers = {
    "Authorization": f"token {TOKEN}",
    "Accept": "application/vnd.github.v3+json",
}

content_b64 = base64.b64encode(out.read_bytes()).decode()

# Get current SHA
r = requests.get(
    f"https://api.github.com/repos/{REPO}/contents/{FILE}?ref={BRANCH}",
    headers=headers,
)
r.raise_for_status()
sha = r.json()["sha"]

# Push
payload = {
    "message": f"dashboard: refresh {snap['as_of']}",
    "content": content_b64,
    "sha": sha,
    "branch": BRANCH,
}
r = requests.put(
    f"https://api.github.com/repos/{REPO}/contents/{FILE}",
    json=payload,
    headers=headers,
)
r.raise_for_status()
print("Pushed:", r.json()["commit"]["sha"])
print("Dashboard will update at https://trevmon28.github.io/macro-dashboard/ in ~60s")